In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
dataset = load_dataset("jmajkutewicz/sql-create-context")

In [2]:
system_prompt = 'Return SQL Query that will answer the question using the CREATE statement as context'

def _create_user_prompt(question: str, context: str) -> str:
    return f"Question: {question}\n\nContext: {context}"

def create_question(e):
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': _create_user_prompt(e['question'], e['context'])},
    ]

def format_text(records):
    # TODO: get questions from the records batch
    questions = records['question']
    # TODO: get contexts from the records batch
    contexts = records['context']
    # TODO: get answers from the records batch
    answers = records['answer']
    
    texts = []
    for question, context, answer in zip(questions, contexts, answers):
        # TODO: create message list for training
        msg = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': _create_user_prompt(question, context)},
            {'role': 'assistant', 'content': answer }
        ]
        # TODO: format the messages into plain text using the
        # tokenizer apply_chat_template. Remember to set tokenize=False
        # and add_generation_prompt=False
        text = tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}
    
max_length = 256
def filter_examples(row):
    # TODO: filter by token length of the formatted ‘text‘ field.
    # tokenized_text_length = len(tokenizer.encode(row['text']))
    tokenized_text_length = len(tokenizer(row["text"])['input_ids'])
    return tokenized_text_length < max_length

answers = []
def test_model(tested_model):
    model_answers = []
    for e in test_dataset:
        q = create_question(e)
        
        inputs = tokenizer.apply_chat_template(
        	q,
        	add_generation_prompt=True,
        	tokenize=True,
        	return_dict=True,
        	return_tensors="pt",
        ).to(tested_model.device)
        outputs = tested_model.generate(**inputs, max_new_tokens=256)
        ans = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
        model_answers.append(ans)
        
        print('# Question:', e['question'])
        print('# Model anwer:\n', ans)
        print('# Ground truth:\n', e['answer'])
        print('-' * 10)
    answers.append(model_answers)

In [3]:
# TODO: get the train split from the dataset
train_dataset = dataset['train']
# TODO: create the training text. Use the .map() method with
# ‘format_text‘ function and set batched=True
train_dataset = train_dataset.map(format_text, batched=True)
# TODO: remove rows that exceed 256 tokens in the formatted ‘text‘ field
train_dataset = train_dataset.filter(filter_examples)
# TODO: limit the train set size to 4096 records
train_dataset = train_dataset.take(4096)
    
# TODO: get the test split from the dataset
test_dataset = dataset['test']
# TODO: limit the test set size to 8 records
test_dataset = test_dataset.take(8)

In [4]:
# Base model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
# Test base model
test_model(model)

# Question: How many other deaths were in the attack with 242 Israelis and/or foreigners wounded?
# Model anwer:
 To answer how many other deaths occurred in an attack involving 242 Israelis or foreigners wounded, you can use the following SQL query:

```sql
SELECT COUNT(other_deaths)
FROM table_name_81
WHERE israeli_and_or_foreigner_wounded = 'yes';
```

This query counts all rows in the `table_name_81` where the value of the column `israeli_and_or_foreigner_wounded` is 'yes', which indicates a death involving both Israeli and foreign nationals.<|im_end|>
# Ground truth:
 SELECT other_deaths FROM table_name_81 WHERE israeli_and_or_foreigner_wounded = "242"
----------
# Question: Which Country has a To par of –9?
# Model anwer:
 SELECT T1.country FROM table_name_33 AS T1 JOIN (
  SELECT country, MIN(to_par) as min_to_par 
  FROM table_name_33 
  GROUP BY country
) AS T2 ON T1.country = T2.country AND T2.min_to_par = -9<|im_end|>
# Ground truth:
 SELECT country FROM table_name_33 WHERE 

In [6]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=8,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", 
                    "o_proj", "gate_proj", "up_proj", "down_proj"]
)

peft_model = get_peft_model(model, peft_config)

In [7]:
peft_model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [8]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    num_train_epochs=5,
    per_device_train_batch_size=32, # TODO: Set batch size
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=0.2,
    optim='adamw_torch',
    dataset_text_field='text', # TODO: Set dataset text field
    max_length=128, # TODO: Set maximum sequence length
    logging_strategy="steps",
    logging_steps=10,
    seed=42,
    push_to_hub=False,
    save_total_limit=1,
    save_strategy="steps",
    save_steps=1024,
    output_dir="tmp",
    report_to=[],
)

trainer = SFTTrainer(
    peft_model,
    train_dataset=train_dataset,
    args=sft_config
)

peft_model.config.use_cache = False

In [9]:
train_result = trainer.train()
print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.509506
20,3.508614
30,3.467558
40,3.392887
50,3.298147
60,3.164682
70,2.923308
80,2.652839
90,2.338352
100,2.090124


TrainOutput(global_step=320, training_loss=1.7477636843919755, metrics={'train_runtime': 366.445, 'train_samples_per_second': 55.888, 'train_steps_per_second': 0.873, 'total_flos': 5699622506741760.0, 'train_loss': 1.7477636843919755})


In [10]:
# Test fine-tuned model
peft_model.eval()
test_model(peft_model)

# Question: How many other deaths were in the attack with 242 Israelis and/or foreigners wounded?
# Model anwer:
 SELECT COUNT(other_deaths) FROM table_name_81 WHERE israeli_and_or_foreigner_wounded = "242"<|im_end|>
# Ground truth:
 SELECT other_deaths FROM table_name_81 WHERE israeli_and_or_foreigner_wounded = "242"
----------
# Question: Which Country has a To par of –9?
# Model anwer:
 SELECT country FROM table_name_33 WHERE to_par = "-9"<|im_end|>
# Ground truth:
 SELECT country FROM table_name_33 WHERE to_par = "–9"
----------
# Question: Which producer did Daniel Cormack direct?
# Model anwer:
 SELECT producer_s_ FROM table_name_50 WHERE director_s_ = "daniel cormack"<|im_end|>
# Ground truth:
 SELECT producer_s_ FROM table_name_50 WHERE director_s_ = "daniel cormack"
----------
# Question: Which away team had a crowd of over 23,000 people?
# Model anwer:
 SELECT away_team FROM table_name_95 WHERE crowd > 23000<|im_end|>
# Ground truth:
 SELECT away_team FROM table_name_95 WHERE

In [16]:
for (q, p1, p2, a) in zip(test_dataset['question'], answers[0], answers[1], test_dataset['answer']):
    print('# Question:', q)
    print('# Base model answer:', p1)
    print('# Fine-tuned model answer:', p2)
    print('Ground truth:', a)
    print('-' * 20, end='\n\n')

# Question: How many other deaths were in the attack with 242 Israelis and/or foreigners wounded?
# Base model answer: To answer how many other deaths occurred in an attack involving 242 Israelis or foreigners wounded, you can use the following SQL query:

```sql
SELECT COUNT(other_deaths)
FROM table_name_81
WHERE israeli_and_or_foreigner_wounded = 'yes';
```

This query counts all rows in the `table_name_81` where the value of the column `israeli_and_or_foreigner_wounded` is 'yes', which indicates a death involving both Israeli and foreign nationals.<|im_end|>
# Fine-tuned model answer: SELECT COUNT(other_deaths) FROM table_name_81 WHERE israeli_and_or_foreigner_wounded = "242"<|im_end|>
Ground truth: SELECT other_deaths FROM table_name_81 WHERE israeli_and_or_foreigner_wounded = "242"
--------------------

# Question: Which Country has a To par of –9?
# Base model answer: SELECT T1.country FROM table_name_33 AS T1 JOIN (
  SELECT country, MIN(to_par) as min_to_par 
  FROM table_name_